In [2]:
import pandas as pd
import numpy as np
from autogluon.tabular import TabularDataset, TabularPredictor

c:\Users\Admin\AppData\Local\Programs\Python\Python310\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
def load_and_process_data(filepath):
    df = pd.read_csv(filepath, parse_dates=['Date'], index_col='Date')
    
    # to make the rolling windows work correctly
    df = df.sort_index()
    
    df['H-L'] = df['High'] - df['Low']
    df['O-C'] = df['Close'] - df['Open']
    df['7_DAYS_MA'] = df['Close'].rolling(window=7).mean()
    df['14_DAYS_MA'] = df['Close'].rolling(window=14).mean()
    df['21_DAYS_MA'] = df['Close'].rolling(window=21).mean()
    df['7_DAYS_STD_DEV'] = df['Close'].rolling(window=7).std()
    df['Target_Close'] = df['Close'].shift(-1)
    
    df = df.dropna()
    
    return df

In [4]:
df = load_and_process_data('../data/GS_stock_data.csv')

df.head()

,Close,High,Low,Open,Volume,H-L,O-C,7_DAY_MA,14_DAY_MA,21_DAY_MA,7_DAY_STD,7_DAYS_MA,14_DAYS_MA,21_DAYS_MA,7_DAYS_STD_DEV,Target_Close
Date,,,,,,,,,,,,,,,,
2009-05-05,101.351753,102.101396,99.365203,99.627572,16811000,2.736193,1.724182,95.706942,92.926298,91.874478,4.272873,95.706942,92.926298,91.874478,4.272873,104.365326
2009-05-06,104.365326,105.219919,101.988958,102.551191,20439300,3.230960,1.814135,97.668867,93.891732,92.680167,4.687069,97.668867,93.891732,92.680167,4.687069,100.249802
2009-05-07,100.249802,106.119510,98.810488,105.684716,23188600,7.309023,-5.434914,99.067491,94.594794,93.310224,3.483388,99.067491,94.594794,93.310224,3.483388,104.642700
2009-05-08,104.642700,104.642700,99.814996,101.119379,19152000,4.827704,3.523321,100.396502,95.910954,94.196946,3.596897,100.396502,95.910954,94.196946,3.596897,101.786552
2009-05-11,101.786552,104.170424,101.074394,102.588676,18391800,3.096030,-0.802124,101.176130,96.736632,94.605679,3.129436,101.176130,96.736632,94.605679,3.129436,101.509193


In [5]:
feature_columns = ['H-L', 'O-C', '7_DAYS_MA', '14_DAYS_MA', 
                   '21_DAYS_MA', '7_DAYS_STD_DEV', 'Volume']
label_column = 'Target_Close'

model_data = df[feature_columns + [label_column]]

In [6]:
# train/test Split 
# training: 04/05/2009 - 04/03/2017
# testing: 04/04/2017 - 04/05/2019
split_date = '2017-04-04'

train_data = model_data.loc[model_data.index < split_date]
test_data = model_data.loc[model_data.index >= split_date]

print(f"Training Data Shape: {train_data.shape}")
print(f"Testing Data Shape: {test_data.shape}")

Training Data Shape: (1993, 8)
Testing Data Shape: (503, 8)


In [7]:
#shape is like in the paper

In [10]:
predictor = TabularPredictor(
    label=label_column,
    problem_type='regression',
    eval_metric='root_mean_squared_error', # Paper uses RMSE, so we do the same
    path='stock_price_ann_model'
).fit(
    train_data,
    # force autogluon to use neural networks
    hyperparameters={
        'NN_TORCH': 
            {'num_epochs': 50, 'learning_rate': 1e-3},
        'FASTAI': {}
    }, 
    time_limit=600
)

Verbosity: 2 (Standard Logging)
=================== System Info ===================
AutoGluon Version:  1.4.0
Python Version:     3.10.11
Operating System:   Windows
Platform Machine:   AMD64
Platform Version:   10.0.19045
CPU Count:          6
Memory Avail:       4.16 GB / 15.92 GB (26.1%)
Disk Space Avail:   36.72 GB / 237.20 GB (15.5%)
No presets specified! To achieve strong results with AutoGluon, it is recommended to use the available presets. Defaulting to `'medium'`...
	Recommended Presets (For more details refer to https://auto.gluon.ai/stable/tutorials/tabular/tabular-essentials.html#presets):
	presets='extreme' : New in v1.4: Massively better than 'best' on datasets <30000 samples by using new models meta-learned on https://tabarena.ai: TabPFNv2, TabICL, Mitra, and TabM. Absolute best accuracy. Requires a GPU. Recommended 64 GB CPU memory and 32+ GB GPU memory.
	presets='best'    : Maximize accuracy. Recommended for most users. Use in competitions and benchmarks.
	presets='hi

RuntimeError: No models were trained successfully during fit(). Inspect the log output or increase verbosity to determine why no models were fit. Alternatively, set `raise_on_no_models_fitted` to False during the fit call.

In [ ]:
# paper uses RMSE and MAPE
performance = predictor.evaluate(test_data)
print("Evaluation Results:")
print(f"RMSE: {performance['root_mean_squared_error']:.4f}")

# might not be the default metric returned
predictions = predictor.predict(test_data)
mape = np.mean(np.abs((test_data[label_column] - predictions) / test_data[label_column])) * 100
print(f"MAPE: {mape:.2f}%")